# Random Forest - potencial solar e eolico com gold

Este notebook usa exclusivamente `data/gold/inmet_pe_daily.csv` para treinar um Random Forest multi-saida que estima potencial esperado diario em kWh. Quando `USE_HISTORICAL_FEATURES=True`, o modelo tambem usa agregados historicos calculados sem vazamento temporal.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import mlflow
import mlflow.sklearn
import pandas as pd
from threadpoolctl import threadpool_limits

from src.modeling.gold_energy import (
    FEATURE_COLUMNS,
    TARGET_COLUMNS,
    assert_no_forbidden_features,
    configure_mlflow_tracking,
    describe_search_space,
    evaluate_predictions,
    fit_final_model,
    load_gold_daily,
    make_future_feature_frame,
    make_temporal_cv_splits,
    prepare_energy_modeling_table,
    split_feature_target_metadata,
    temporal_train_test_split,
    to_jsonable,
    train_random_search,
    grid_size,
    random_forest_base_pipeline,
    random_forest_refinement_grid,
    random_forest_search_space,
    train_grid_search,
)
from src.modeling.historical_features import (
    ClimatologyBaselineRegressor,
    HISTORICAL_FEATURE_COLUMNS,
    compare_metric_tables,
    fit_historical_feature_reference,
    prepare_historical_train_test_frames,
    save_historical_reference,
    split_feature_target_metadata_with_columns,
    transform_with_historical_features,
)
from src.modeling.training_config import (
    BLAS_THREADS,
    CPU_WORKERS,
    ENERGY_CONFIG,
    FUTURE_DATE,
    FUTURE_STATION_CODE,
    HISTORY_MIN_OBSERVATIONS_DAY,
    HISTORY_MIN_OBSERVATIONS_MONTH,
    MLFLOW_EXPERIMENT_NAME,
    PRODUCTION_REFIT_WITH_FULL_GOLD,
    TEST_YEAR_FRACTION,
    USE_HISTORICAL_FEATURES,
)

RUN_ARTIFACTS_DIR = configure_mlflow_tracking(PROJECT_ROOT, MLFLOW_EXPERIMENT_NAME)

print(f"Raiz do projeto: {PROJECT_ROOT}")
print(f"Diretorio de artefatos de modelagem: {RUN_ARTIFACTS_DIR}")

In [ ]:
# Configuracoes compartilhadas entre modelos ficam em src/modeling/training_config.py.
MODEL_NAME = "random_forest"
RANDOM_STATE = 42
N_ITER_RANDOM = 40

assert CPU_WORKERS > 0, "CPU_WORKERS deve ser positivo."
assert BLAS_THREADS > 0, "BLAS_THREADS deve ser positivo."
print(f"CPU_WORKERS={CPU_WORKERS}; BLAS_THREADS={BLAS_THREADS}; threads planejadas={CPU_WORKERS * BLAS_THREADS}")
print(f"USE_HISTORICAL_FEATURES={USE_HISTORICAL_FEATURES}")

In [ ]:
daily_gold = load_gold_daily(PROJECT_ROOT)
modeling_table = prepare_energy_modeling_table(daily_gold, ENERGY_CONFIG)
X_base, y_base, metadata_base = split_feature_target_metadata(modeling_table)

(
    X_train_base,
    X_test_base,
    y_train_base,
    y_test_base,
    train_metadata_base,
    test_metadata_base,
    train_years,
    test_years,
) = temporal_train_test_split(X_base, y_base, metadata_base, test_year_fraction=TEST_YEAR_FRACTION)

baseline_model = ClimatologyBaselineRegressor(
    min_observations_day=HISTORY_MIN_OBSERVATIONS_DAY,
    min_observations_month=HISTORY_MIN_OBSERVATIONS_MONTH,
)
X_train_base_for_baseline = X_train_base.copy()
X_train_base_for_baseline["year"] = train_metadata_base["year"].to_numpy()
baseline_model.fit(X_train_base_for_baseline, y_train_base)
baseline_predictions = baseline_model.predict(X_test_base)
baseline_metrics_table, baseline_metrics = evaluate_predictions(y_test_base, baseline_predictions)

if USE_HISTORICAL_FEATURES:
    train_frame, test_frame, train_history_reference = prepare_historical_train_test_frames(
        modeling_table,
        train_years,
        test_years,
        min_observations_day=HISTORY_MIN_OBSERVATIONS_DAY,
        min_observations_month=HISTORY_MIN_OBSERVATIONS_MONTH,
    )
    model_feature_columns = FEATURE_COLUMNS + HISTORICAL_FEATURE_COLUMNS
    X_train, y_train, train_metadata = split_feature_target_metadata_with_columns(train_frame, model_feature_columns)
    X_test, y_test, test_metadata = split_feature_target_metadata_with_columns(test_frame, model_feature_columns)
else:
    train_history_reference = None
    model_feature_columns = FEATURE_COLUMNS
    X_train = X_train_base
    X_test = X_test_base
    y_train = y_train_base
    y_test = y_test_base
    train_metadata = train_metadata_base
    test_metadata = test_metadata_base

assert_no_forbidden_features(model_feature_columns)
cv_splits = make_temporal_cv_splits(train_metadata)
effective_train_years = sorted(train_metadata["year"].dropna().astype(int).unique().tolist())

print(f"Linhas gold usadas: {len(modeling_table):,}")
print(f"Linhas de treino do modelo: {len(X_train):,}")
print(f"Linhas de teste final: {len(X_test):,}")
print(f"Variaveis de entrada: {model_feature_columns}")
print(f"Alvos: {TARGET_COLUMNS}")
print(f"Anos treino/validacao originais: {train_years}")
print(f"Anos treino efetivos do modelo: {effective_train_years}")
print(f"Anos teste final: {test_years}")
print(f"Folds temporais no treino: {len(cv_splits)}")
print("Metricas da baseline no mesmo teste temporal:")
display(baseline_metrics_table)

In [ ]:
rf_pipeline = random_forest_base_pipeline(random_state=RANDOM_STATE, feature_columns=model_feature_columns)
rf_space = random_forest_search_space()
print("Espaco amplo RF:", describe_search_space(rf_space))

mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

with mlflow.start_run(run_name="random_forest_gold_energy_hist"):
    mlflow_run_id = mlflow.active_run().info.run_id
    mlflow.log_param("model_family", "RandomForestRegressor")
    mlflow.log_param("cpu_workers", CPU_WORKERS)
    mlflow.log_param("blas_threads", BLAS_THREADS)
    mlflow.log_param("random_state", RANDOM_STATE)
    mlflow.log_param("n_iter_random", N_ITER_RANDOM)
    mlflow.log_param("use_historical_features", USE_HISTORICAL_FEATURES)
    mlflow.log_param("history_min_observations_day", HISTORY_MIN_OBSERVATIONS_DAY)
    mlflow.log_param("history_min_observations_month", HISTORY_MIN_OBSERVATIONS_MONTH)
    mlflow.log_param("train_years", ",".join(map(str, train_years)))
    mlflow.log_param("effective_train_years", ",".join(map(str, effective_train_years)))
    mlflow.log_param("test_years", ",".join(map(str, test_years)))
    mlflow.log_param("features", ",".join(model_feature_columns))
    mlflow.log_dict(to_jsonable(ENERGY_CONFIG), "energy_config.json")
    mlflow.log_dict(to_jsonable(rf_space), "search_space_random.json")
    for metric_name, metric_value in baseline_metrics.items():
        if pd.notna(metric_value):
            mlflow.log_metric(f"baseline_{metric_name}", float(metric_value))

    with threadpool_limits(limits=BLAS_THREADS):
        broad_search, broad_seconds = train_random_search(
            rf_pipeline,
            rf_space,
            X_train,
            y_train,
            cv_splits,
            n_iter=N_ITER_RANDOM,
            n_jobs=CPU_WORKERS,
            random_state=RANDOM_STATE,
        )

    rf_ref_grid = random_forest_refinement_grid(broad_search.best_params_)
    print(f"Grid de refino RF: {grid_size(rf_ref_grid)} combinacoes")
    mlflow.log_metric("broad_search_seconds", broad_seconds)
    mlflow.log_metric("broad_best_balanced_negative_nrmse", broad_search.best_score_)
    mlflow.log_dict(to_jsonable(broad_search.best_params_), "best_params_broad.json")
    mlflow.log_dict(to_jsonable(rf_ref_grid), "search_space_refinement.json")

    with threadpool_limits(limits=BLAS_THREADS):
        refine_search, refine_seconds = train_grid_search(
            rf_pipeline,
            rf_ref_grid,
            X_train,
            y_train,
            cv_splits,
            n_jobs=CPU_WORKERS,
        )

    final_params = dict(refine_search.best_params_)
    final_params["model__n_jobs"] = CPU_WORKERS
    with threadpool_limits(limits=BLAS_THREADS):
        final_model, final_fit_seconds = fit_final_model(rf_pipeline, final_params, X_train, y_train)


    test_predictions = final_model.predict(X_test)
    metrics_table, final_metrics = evaluate_predictions(y_test, test_predictions)
    comparison_table = compare_metric_tables(metrics_table, baseline_metrics_table, MODEL_NAME)
    for metric_name, metric_value in final_metrics.items():
        if pd.notna(metric_value):
            mlflow.log_metric(metric_name, float(metric_value))
    mlflow.log_metric("refine_search_seconds", refine_seconds)
    mlflow.log_metric("final_fit_seconds", final_fit_seconds)

    mlflow.log_metric("refine_best_balanced_negative_nrmse", refine_search.best_score_)
    mlflow.log_dict(to_jsonable(refine_search.best_params_), "best_params_refinement.json")

    mlflow.sklearn.log_model(final_model, artifact_path="model")


search_summary = pd.DataFrame(
    [
        {"fase": "busca_ampla", "balanced_negative_nrmse": broad_search.best_score_, "segundos": broad_seconds},
        {"fase": "refino", "balanced_negative_nrmse": refine_search.best_score_, "segundos": refine_seconds},
        {"fase": "treino_final", "balanced_negative_nrmse": None, "segundos": final_fit_seconds},
    ]
)
best_params_table = pd.Series(refine_search.best_params_, name="valor").rename_axis("hiperparametro").reset_index()


print("RESULTADOS FINAIS - RANDOM FOREST")
print(f"MLflow run_id: {mlflow_run_id}")
print(f"Anos de treino/validacao: {train_years}")
print(f"Anos de treino efetivos: {effective_train_years}")
print(f"Anos de teste final: {test_years}")
print("Metricas no teste temporal final:")
display(metrics_table)
print("Comparacao contra baseline:")
display(comparison_table)
print("Resumo da busca de hiperparametros:")
display(search_summary)
print("Melhores hiperparametros do refino:")
display(best_params_table)

In [ ]:
# Resumo explicito das metricas de erro continuo e aderencia no teste temporal final.
print("RESUMO DAS METRICAS - TESTE TEMPORAL FINAL")
print(f"Modelo: {MODEL_NAME}")
print(f"MLflow run_id: {mlflow_run_id}")
print(f"Anos de teste final: {test_years}")
print(f"balanced_nrmse: {final_metrics['balanced_nrmse']:.6f}")
print(f"baseline_balanced_nrmse: {baseline_metrics['balanced_nrmse']:.6f}")
print("")
for _, row in metrics_table.iterrows():
    print(f"Alvo: {row['target']}")
    print(f"  MAE: {row['mae']:.6f}")
    print(f"  RMSE: {row['rmse']:.6f}")
    print(f"  NRMSE: {row['nrmse']:.6f}")
    print(f"  R2: {row['r2']:.6f}")
    print(f"  Bias medio: {row['bias']:.6f}")
    print(f"  MedAE: {row['medae']:.6f}")
    print(f"  sMAPE (%): {row['smape']:.6f}")

print("")
print("Metricas usadas na busca de hiperparametros:")
print(f"  broad_best_balanced_negative_nrmse: {broad_search.best_score_:.6f}")
print(f"  refine_best_balanced_negative_nrmse: {refine_search.best_score_:.6f}")
print(f"  broad_search_seconds: {broad_seconds:.2f}")
print(f"  refine_search_seconds: {refine_seconds:.2f}")
print(f"  final_fit_seconds: {final_fit_seconds:.2f}")

print("")
print("Comparacao contra baseline:")
display(comparison_table)

In [ ]:
test_results = test_metadata.copy()
for index, target in enumerate(TARGET_COLUMNS):
    test_results[f"{target}_actual"] = y_test[target].to_numpy()
    test_results[f"{target}_pred"] = test_predictions[:, index]
    test_results[f"{target}_baseline"] = baseline_predictions[:, index]
test_results["hybrid_generation_kwh_day_actual"] = test_results[[f"{target}_actual" for target in TARGET_COLUMNS]].sum(axis=1)
test_results["hybrid_generation_kwh_day_pred"] = test_results[[f"{target}_pred" for target in TARGET_COLUMNS]].sum(axis=1)
test_results["hybrid_generation_kwh_day_baseline"] = test_results[[f"{target}_baseline" for target in TARGET_COLUMNS]].sum(axis=1)

results_dir = RUN_ARTIFACTS_DIR / "evaluation" / MODEL_NAME
results_dir.mkdir(parents=True, exist_ok=True)
run_timestamp = pd.Timestamp.now().strftime("%Y%m%d-%H%M%S")
metrics_output_path = results_dir / f"{MODEL_NAME}_metrics_{run_timestamp}.csv"
baseline_metrics_output_path = results_dir / f"{MODEL_NAME}_baseline_metrics_{run_timestamp}.csv"
comparison_output_path = results_dir / f"{MODEL_NAME}_baseline_comparison_{run_timestamp}.csv"
predictions_output_path = results_dir / f"{MODEL_NAME}_test_predictions_{run_timestamp}.csv"
predictions_sample_output_path = results_dir / f"{MODEL_NAME}_test_predictions_sample_{run_timestamp}.csv"

metrics_table.to_csv(metrics_output_path, index=False)
baseline_metrics_table.to_csv(baseline_metrics_output_path, index=False)
comparison_table.to_csv(comparison_output_path, index=False)
test_results.to_csv(predictions_output_path, index=False)
test_results.head(50).to_csv(predictions_sample_output_path, index=False)

with mlflow.start_run(run_id=mlflow_run_id):
    mlflow.log_artifact(str(metrics_output_path), artifact_path="evaluation")
    mlflow.log_artifact(str(baseline_metrics_output_path), artifact_path="evaluation")
    mlflow.log_artifact(str(comparison_output_path), artifact_path="evaluation")
    mlflow.log_artifact(str(predictions_sample_output_path), artifact_path="evaluation")
    if train_history_reference is not None:
        train_reference_dir = results_dir / "historical_reference_train"
        production_reference = (
            fit_historical_feature_reference(
                modeling_table,
                min_observations_day=HISTORY_MIN_OBSERVATIONS_DAY,
                min_observations_month=HISTORY_MIN_OBSERVATIONS_MONTH,
            )
            if PRODUCTION_REFIT_WITH_FULL_GOLD
            else train_history_reference
        )
        production_reference_dir = results_dir / "historical_reference_production"
        save_historical_reference(train_history_reference, train_reference_dir)
        save_historical_reference(production_reference, production_reference_dir)
        mlflow.log_artifacts(str(train_reference_dir), artifact_path="historical_reference_train")
        mlflow.log_artifacts(str(production_reference_dir), artifact_path="historical_reference_production")

print("Arquivos de resultados salvos:")
print(f"- Metricas: {metrics_output_path}")
print(f"- Metricas da baseline: {baseline_metrics_output_path}")
print(f"- Comparacao com baseline: {comparison_output_path}")
print(f"- Predicoes completas do teste: {predictions_output_path}")
print(f"- Amostra das predicoes: {predictions_sample_output_path}")
print("Primeiras predicoes do teste temporal:")
display(test_results.head(20))

In [ ]:
# Preencha FUTURE_STATION_CODE e FUTURE_DATE em src/modeling/training_config.py.
if FUTURE_STATION_CODE is None or FUTURE_DATE is None:
    print("Preencha FUTURE_STATION_CODE e FUTURE_DATE para gerar inferencia futura.")
else:
    future_frame = make_future_feature_frame(modeling_table, FUTURE_DATE)
    if USE_HISTORICAL_FEATURES:
        future_reference = (
            fit_historical_feature_reference(
                modeling_table,
                min_observations_day=HISTORY_MIN_OBSERVATIONS_DAY,
                min_observations_month=HISTORY_MIN_OBSERVATIONS_MONTH,
            )
            if PRODUCTION_REFIT_WITH_FULL_GOLD
            else train_history_reference
        )
        future_frame = transform_with_historical_features(future_frame, future_reference)
    future_predictions = final_model.predict(future_frame[model_feature_columns])
    future_ranking = future_frame[[
        column
        for column in ["station_code", "station_name", "city", "state", "latitude", "longitude", "altitude", "date"]
        if column in future_frame.columns
    ]].copy()
    future_ranking["solar_generation_kwh_day_pred"] = future_predictions[:, 0]
    future_ranking["wind_generation_kwh_day_pred"] = future_predictions[:, 1]
    future_ranking["hybrid_generation_kwh_day_pred"] = (
        future_ranking["solar_generation_kwh_day_pred"] + future_ranking["wind_generation_kwh_day_pred"]
    )
    future_ranking = future_ranking.sort_values("hybrid_generation_kwh_day_pred", ascending=False).reset_index(drop=True)

    station_code = str(FUTURE_STATION_CODE)
    station_prediction = future_ranking[future_ranking["station_code"].astype(str) == station_code].copy()
    if station_prediction.empty:
        known = ", ".join(future_ranking["station_code"].astype(str).head(10).tolist())
        raise ValueError(f"station_code nao encontrado na gold: {station_code}. Exemplos conhecidos: {known}")
    station_prediction.insert(0, "ranking_position", station_prediction.index + 1)
    display(station_prediction.reset_index(drop=True))
    display(future_ranking.head(20))